# Validação 17 — Orquestração de múltiplos artigos

## Objetivo

Validar o primeiro fluxo integrado após a etapa 15: partir somente de uma alegação, criar uma consulta científica, buscar candidatos no PubMed, processar múltiplos artigos independentes e gerar uma única resposta explicável.

O teste elimina o PMID fixo usado nas validações anteriores. A unidade de síntese passa a ser o **artigo**, não o trecho, para evitar que muitos trechos de uma mesma publicação sejam confundidos com várias evidências independentes.

## Setup

Este notebook faz chamadas reais ao PubMed, PubMed Central, Crossref, DataCite e ClinicalTrials.gov. Os modelos locais de embeddings e NLI são carregados uma única vez e reutilizados entre os artigos.

Critério de sucesso: obter pelo menos dois artigos processados, preservar eventuais falhas por candidato e produzir um relatório com proveniência para todos os artigos sintetizados.

In [1]:
from pathlib import Path
import os
import sys

from IPython.display import Markdown, display

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from fatofake import (
    ClinicalTrialsClient,
    CrossrefClient,
    DataCiteClient,
    EvidenceStrength,
    MultiArticleAnalysisConfig,
    MultiArticleAnalysisService,
    PmcClient,
    PubMedClient,
    ScientificArticleProcessor,
    SentenceTransformerEncoder,
    TransformersNliClassifier,
    render_evidence_report_markdown,
)


def display_table(rows, columns):
    def clean(value):
        return str(value).replace('|', r'\|').replace('\n', ' ')

    header = '| ' + ' | '.join(columns) + ' |'
    separator = '| ' + ' | '.join('---' for _ in columns) + ' |'
    body = [
        '| ' + ' | '.join(clean(row.get(column, '')) for column in columns) + ' |'
        for row in rows
    ]
    display(Markdown('\n'.join([header, separator, *body])))

## Etapas

### 1. Gerar a consulta a partir da alegação

O planejador abaixo representa a política de busca desta prova de conceito para o domínio estudado. Ele recebe a alegação, confirma que os conceitos esperados estão presentes e gera uma consulta temática — nunca um identificador de artigo.

In [2]:
class CoffeeProstateQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        normalized = claim.casefold()
        if 'café' not in normalized and 'cafe' not in normalized:
            raise ValueError('A alegação precisa mencionar café para este planejador de demonstração.')
        if 'próstata' not in normalized and 'prostata' not in normalized:
            raise ValueError('A alegação precisa mencionar próstata para este planejador de demonstração.')
        return ['coffee[Title] AND (prostate[Title] OR prostatic[Title])']


claim = 'Beber café pode alterar o risco de câncer de próstata.'
planner = CoffeeProstateQueryPlanner()
queries = planner.generate_queries(claim)

display_table(
    [{'alegação': claim, 'consulta PubMed': queries[0]}],
    ['alegação', 'consulta PubMed'],
)
assert all('[pmid]' not in query.casefold() for query in queries)

| alegação | consulta PubMed |
| --- | --- |
| Beber café pode alterar o risco de câncer de próstata. | coffee[Title] AND (prostate[Title] OR prostatic[Title]) |

### 2. Montar o processador e o serviço

O `ScientificArticleProcessor` executa o pipeline científico dentro de cada publicação. O `MultiArticleAnalysisService` coordena a busca, avança para o próximo candidato quando uma falha conhecida ocorre, exige independência por PMID e só sintetiza depois de alcançar o mínimo configurado.

In [3]:
ncbi_email = os.getenv('NCBI_EMAIL')
ncbi_api_key = os.getenv('NCBI_API_KEY')

embedding_encoder = SentenceTransformerEncoder()
nli_classifier = TransformersNliClassifier()

processor = ScientificArticleProcessor(
    pmc_client=PmcClient(email=ncbi_email, api_key=ncbi_api_key),
    embedding_encoder=embedding_encoder,
    nli_classifier=nli_classifier,
    crossref_client=CrossrefClient(email=os.getenv('CROSSREF_EMAIL')),
    datacite_client=DataCiteClient(),
    clinical_trials_client=ClinicalTrialsClient(),
    hybrid_top_k=8,
)

config = MultiArticleAnalysisConfig(
    max_results_per_query=8,
    target_articles=3,
    minimum_successful_articles=2,
)
service = MultiArticleAnalysisService(
    query_planner=planner,
    pubmed_client=PubMedClient(email=ncbi_email, api_key=ncbi_api_key),
    article_processor=processor,
    config=config,
)

print(f'Modelo de embeddings: {embedding_encoder.name}')
print(f'Modelo NLI: {nli_classifier.name}')
print(f'Alvo: {config.target_articles} artigos; mínimo: {config.minimum_successful_articles}.')

/private/tmp/fatofake-notebook-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23801.37it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8953.62it/s]

Modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Modelo NLI: MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli
Alvo: 3 artigos; mínimo: 2.


### 3. Executar a análise integrada

Uma única chamada percorre o fluxo completo. O objeto retornado mantém a entrada normalizada, o plano de busca, os candidatos, os artigos aceitos, as falhas, a síntese e o relatório final.

In [4]:
analysis = service.analyze(claim)
print(
    f'Análise concluída: {len(analysis.articles)} artigo(s) incluído(s) e '
    f'{len(analysis.failures)} falha(s) registrada(s).'
)

Análise concluída: 3 artigo(s) incluído(s) e 0 falha(s) registrada(s).


### 4. Inspecionar cobertura, artigos e falhas

As tabelas tornam o comportamento verificável: distinguem o universo recuperado dos artigos que efetivamente chegaram à síntese e registram por que algum candidato foi descartado.

In [5]:
query_rows = [
    {'consulta': item.query, 'resultados PubMed': item.total_matches}
    for item in analysis.search_result.query_results
]
display(Markdown('#### Cobertura da busca'))
display_table(query_rows, ['consulta', 'resultados PubMed'])

article_rows = []
for bundle in analysis.articles:
    article_summary = next(
        item for item in analysis.synthesis.articles
        if item.pmid == bundle.publication.pmid
    )
    article_rows.append({
        'PMID': bundle.publication.pmid,
        'título': bundle.publication.title,
        'acesso': bundle.content.access_level,
        'desenho': bundle.quality_report.study_design.value,
        'qualidade': bundle.quality_report.quality_level.value,
        'evidências': len(bundle.assessments),
        'direção': article_summary.direction.value,
    })

display(Markdown('#### Artigos incluídos'))
display_table(
    article_rows,
    ['PMID', 'título', 'acesso', 'desenho', 'qualidade', 'evidências', 'direção'],
)

failure_rows = [
    {'PMID': item.pmid, 'título': item.title, 'etapa': item.stage, 'motivo': item.reason}
    for item in analysis.failures
]
display(Markdown('#### Candidatos não processados'))
if failure_rows:
    display_table(failure_rows, ['PMID', 'título', 'etapa', 'motivo'])
else:
    display(Markdown('Nenhum candidato falhou nesta execução.'))

#### Cobertura da busca

| consulta | resultados PubMed |
| --- | --- |
| coffee[Title] AND (prostate[Title] OR prostatic[Title]) | 43 |

#### Artigos incluídos

| PMID | título | acesso | desenho | qualidade | evidências | direção |
| --- | --- | --- | --- | --- | --- | --- |
| 18985656 | Tea, coffee and prostate cancer. | ABSTRACT_ONLY | UNKNOWN | UNCLEAR | 4 | NEUTRAL |
| 33431520 | Coffee consumption and risk of prostate cancer: a systematic review and meta-analysis. | FULL_TEXT | SYSTEMATIC_REVIEW_META_ANALYSIS | UNCLEAR | 6 | SUPPORTS |
| 24584929 | Coffee consumption and prostate cancer risk: an updated meta-analysis. | ABSTRACT_ONLY | OBSERVATIONAL | UNCLEAR | 6 | NEUTRAL |

#### Candidatos não processados

Nenhum candidato falhou nesta execução.

### 5. Exibir a síntese e a resposta ao usuário

A força descreve o conjunto recuperado e não uma probabilidade de verdade. O relatório explicita limitações e mantém links para auditoria humana.

In [6]:
summary = [{
    'artigos independentes': analysis.synthesis.article_count,
    'direção': analysis.synthesis.direction.value,
    'força': analysis.synthesis.strength.value,
    'há conflito': analysis.synthesis.has_conflict,
    'conclusão comunicada': analysis.report.conclusion.value,
}]
display_table(
    summary,
    ['artigos independentes', 'direção', 'força', 'há conflito', 'conclusão comunicada'],
)
display(Markdown(render_evidence_report_markdown(analysis.report)))

| artigos independentes | direção | força | há conflito | conclusão comunicada |
| --- | --- | --- | --- | --- |
| 3 | MIXED | LOW | False | INCONCLUSIVE |

## Conclusão: Resultado inconclusivo

O conjunto analisado não apresenta direção suficientemente clara.

### O que as evidências sugerem

- Direção do sinal: **sinais mistos**.
- Probabilidades médias do classificador textual — não representam a probabilidade de a alegação estar correta: apoio 42.3%, contradição 7.9% e neutro 49.8%.
- Artigos independentes analisados: **3**.

### Confiança da análise

- Força do conjunto: **baixa**.
- PMID 18985656: confiança metodológica **não esclarecida** pelo módulo de validação externa; qualidade usada na síntese: **não esclarecida**.
- PMID 33431520: confiança metodológica **não esclarecida** pelo módulo de validação externa; qualidade usada na síntese: **não esclarecida**.
- PMID 24584929: confiança metodológica **não esclarecida** pelo módulo de validação externa; qualidade usada na síntese: **não esclarecida**.

### Por que a confiança é limitada

- Perfil preliminar: ainda é necessária uma avaliação estruturada de risco de viés adequada ao desenho do estudo.
- A busca é limitada aos primeiros 8 resultados por consulta e não constitui revisão sistemática.
- Os perfis de qualidade são preliminares e não substituem uma avaliação humana de risco de viés.

### Fontes

- [PMID 18985656: Tea, coffee and prostate cancer.](https://pubmed.ncbi.nlm.nih.gov/18985656/)
- [DOI do PMID 18985656](https://doi.org/10.1002/mnfr.200800218)
- [PMID 33431520: Coffee consumption and risk of prostate cancer: a systematic review and meta-analysis.](https://pubmed.ncbi.nlm.nih.gov/33431520/)
- [Texto completo do PMID 33431520](https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/)
- [DOI do PMID 33431520](https://doi.org/10.1136/bmjopen-2020-038902)
- [PMID 24584929: Coffee consumption and prostate cancer risk: an updated meta-analysis.](https://pubmed.ncbi.nlm.nih.gov/24584929/)
- [DOI do PMID 24584929](https://doi.org/10.1007/s10552-014-0364-8)

## Verificações automáticas

Estas asserções transformam as decisões arquiteturais da etapa em critérios reproduzíveis.

In [7]:
included_pmids = [bundle.publication.pmid for bundle in analysis.articles]
synthesis_pmids = [item.pmid for item in analysis.synthesis.articles]
source_pmids = {source.pmid for source in analysis.report.sources if source.pmid}
failed_pmids = {item.pmid for item in analysis.failures}

checks = {
    'consulta sem PMID fixo': all('[pmid]' not in query.casefold() for query in analysis.search_plan.queries),
    'mínimo de dois artigos': analysis.synthesis.article_count >= 2,
    'PMIDs independentes': len(included_pmids) == len(set(included_pmids)),
    'síntese corresponde aos incluídos': set(included_pmids) == set(synthesis_pmids),
    'força não é insuficiente': analysis.synthesis.strength is not EvidenceStrength.INSUFFICIENT,
    'todos os artigos têm fonte': set(synthesis_pmids).issubset(source_pmids),
    'falhas ficaram fora da síntese': failed_pmids.isdisjoint(synthesis_pmids),
}

display_table(
    [{'verificação': name, 'resultado': 'OK' if passed else 'FALHOU'} for name, passed in checks.items()],
    ['verificação', 'resultado'],
)
assert all(checks.values())
print('Todas as verificações passaram.')

| verificação | resultado |
| --- | --- |
| consulta sem PMID fixo | OK |
| mínimo de dois artigos | OK |
| PMIDs independentes | OK |
| síntese corresponde aos incluídos | OK |
| força não é insuficiente | OK |
| todos os artigos têm fonte | OK |
| falhas ficaram fora da síntese | OK |

Todas as verificações passaram.


## Conclusão

A etapa valida a integração do pipeline em um serviço multiartigo: a alegação gera uma busca temática, diferentes publicações são processadas pela mesma sequência científica e a síntese final contabiliza artigos independentes. Falhas isoladas deixam de interromper automaticamente toda a análise e passam a ser registradas como limitação auditável.

O resultado continua conservador. A busca é limitada, a avaliação de qualidade ainda é preliminar e o relatório não substitui uma revisão sistemática nem avaliação clínica. A próxima etapa é expor este serviço por uma API HTTP, com contrato de entrada, estados de processamento e resposta estruturada para a interface.